In [ ]:
from rdkit.Chem import PandasTools
import numpy as np
import pandas as pd
from rdkit import DataStructs
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle
import random
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.model_selection import LeaveOneOut
from sklearn import preprocessing
# from genetic_selection import GeneticSelectionCV
from mordred import Calculator, descriptors

In [ ]:
data = pd.read_csv("../data/MLdata.csv")

In [ ]:
data

In [ ]:
X = data.iloc[:, 3:]

In [ ]:
X

In [ ]:
y = data.iloc[:, [1,2]]

In [ ]:
y

In [ ]:
def genetic_feature_selection(X, y, estimator=None, **kwargs):
    """
    Runs genetic feature selection
    
    Parameters:
    -----------
    X : pandas.DataFrame
        Feature matrix
    y : array-like
        Target variable
    estimator : object
        Base estimator for scoring features
    **kwargs : dict
        Extra kwargs for GeneticSelectionCV
    
    Returns:
    --------
    selected_features : list
        List of selected features
    model : fitted GeneticSelectionCV model
        Fitted model
    """
    
    if GeneticSelectionCV is None:
        raise ImportError("GeneticSelectionCV is not available. Please install sklearn-genetic-opt")
    
    if estimator is None:
        estimator = RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    
    default_params = {
        'cv': 5,
        'verbose': 3,
        'scoring': "neg_mean_squared_error",
        'max_features': 10,
        'n_population': 100,
        'crossover_proba': 0.5,
        'mutation_proba': 0.2,
        'n_generations': 30,
        'crossover_independent_proba': 0.5,
        'mutation_independent_proba': 0.04,
        'tournament_size': 3,
        'n_gen_no_change': 10,
        'caching': True,
        'n_jobs': -1
    }
    
    default_params.update(kwargs)
    
    model = GeneticSelectionCV(estimator, **default_params)
    
    print("Starting genetic feature selection...")
    print(f"Initial number of features: {X.shape[1]}")
    print(f"Target number of features: {default_params['max_features']}")
    
    model = model.fit(X, y)
    
    selected_features = list(X.columns[model.support_])
    
    print(f"\n✓ Genetic selection completed!")
    print(f"Selected {len(selected_features)} features out of {X.shape[1]}")
    print('Selected Features:', selected_features)
    
    return selected_features, model

In [ ]:
y.iloc[:, 0]

In [ ]:
y

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn_genetic import GAFeatureSelectionCV

from sklearn_genetic.space import Categorical, Integer, Continuous
from sklearn_genetic.plots import plot_fitness_evolution
import matplotlib.pyplot as plt

# y = y.iloc[:, 0]

# rg = RandomForestRegressor(
#     random_state=42,
#     n_jobs=-1
# )

# model = GAFeatureSelectionCV(
#     estimator=rg,
#     scoring="neg_mean_squared_error",
#     elitism=True,
#     verbose=True,
# )
et = ExtraTreesRegressor(
    n_estimators=50,
    min_samples_split=5,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

model = GAFeatureSelectionCV(
    estimator=et,
    cv=3,
    scoring="neg_mean_squared_error",
    population_size=50,
    generations=20,
    features_per_individual=10,
    crossover_probability=0.6,
    mutation_probability=0.3,
    tournament_size=2,
    elitism=True,
    verbose=True,
    n_jobs=1
)
print("Starting genetic feature selection for small dataset...")
print(f"Data shape: {X.shape}")
print(f"Target variable range: {y.min():.3f} to {y.max():.3f}")

model = model.fit(X, y)

In [ ]:
from sklearn_genetic import GAFeatureSelectionCV
import inspect

In [ ]:
def extra_trees_genetic_selection(X, y, n_features=10):
    """
    Genetic feature selection with Extra Trees
    """
    print("=== EXTRA TREES GENETIC FEATURE SELECTION ===")
    print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
    
    et = ExtraTreesRegressor(
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )
    
    model = GAFeatureSelectionCV(
        estimator=et,
        cv=4,
        scoring="neg_mean_squared_error",
        population_size=100,
        generations=20,
        max_features=n_features,
        crossover_probability=0.7,
        mutation_probability=0.3,
        tournament_size=2,
        elitism=True,
        verbose=True,
        n_jobs=1,
        error_score='raise' 
    )

    print("\nStarting Extra Trees genetic selection...")
    model.fit(X, y)
    
    selected_features = X.columns[model.support_].tolist()
    
    print(f"\n✓ Selection completed!")
    print(f"Selected {len(selected_features)} features:")
    for i, feat in enumerate(selected_features, 1):
        print(f"  {i:2d}. {feat}")
    
    return model, selected_features

In [ ]:
print("GAFeatureSelectionCV parameters:")
sig = inspect.signature(GAFeatureSelectionCV.__init__)
for param_name, param in sig.parameters.items():
    if param_name != 'self':
        print(f"  - {param_name}")

In [ ]:
model, features = extra_trees_genetic_selection(X, y, n_features=300)

In [ ]:
print("\n=== RESULTS ===")
print('Selected Features:', list(X.columns[model.support_]))
print(f'Number of selected features: {np.sum(model.support_)}')

In [ ]:
len(X.columns[model.support_])

In [ ]:
features

In [ ]:
X[features]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def create_extra_trees():
    """Builds a conservative Extra Trees model for small samples."""
    return ExtraTreesRegressor(
        n_estimators=30,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.3,
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )

def compare_models_for_small_data(X, y, test_size=0.2):
    """
    Расширенное сравнение моделей для маленькой выборки с полным анализом
    """
    print("=" * 70)
    print("EXTENDED MODEL COMPARISON (SMALL SAMPLE)")
    print("=" * 70)
    print(f"Размер данных: {X.shape[0]} samples × {X.shape[1]} features")
    print(f"Target variable: min={y.min():.3f}, max={y.max():.3f}, mean={y.mean():.3f}")
    
    models = {
        'Ridge Regression': Ridge(alpha=1.0, random_state=42),
        'Lasso Regression': Lasso(alpha=0.1, random_state=42, max_iter=1000),
        'SVR': SVR(kernel='linear', C=1.0),
        'Random Forest ': RandomForestRegressor(
            n_estimators=30, max_depth=4, min_samples_split=5, random_state=42
        ),
        'Extra Trees': create_extra_trees()
    }
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    results = {}
    
    print("\n" + "=" * 70)
    print("CROSS-VALIDATION RESULTS (3-fold CV)")
    print("=" * 70)
    
    for name, model in models.items():
        try:
            if name in ['SVR', 'Ridge Regression', 'Lasso Regression']:
                X_cv = X_train_scaled
            else:
                X_cv = X_train
            
            cv_scores = cross_val_score(model, X_cv, y_train, cv=3, 
                                      scoring='neg_mean_squared_error')
            
            if name in ['SVR', 'Ridge Regression', 'Lasso Regression']:
                model.fit(X_train_scaled, y_train)
                y_pred = model.predict(X_test_scaled)
            else:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
            
            mse_test = mean_squared_error(y_test, y_pred)
            mae_test = mean_absolute_error(y_test, y_pred)
            r2_test = r2_score(y_test, y_pred)
            
            results[name] = {
                'cv_mean': np.mean(cv_scores),
                'cv_std': np.std(cv_scores),
                'cv_scores': cv_scores,
                'mse_test': mse_test,
                'mae_test': mae_test,
                'r2_test': r2_test,
                'model': model,
                'y_pred': y_pred,
                'y_test': y_test
            }
            
            print(f"{name:25} | CV: {np.mean(cv_scores):8.4f} ± {np.std(cv_scores):.4f} | "
                  f"Test MSE: {mse_test:8.4f} | R²: {r2_test:6.3f}")
                  
        except Exception as e:
            print(f"{name:25} | Error: {e}")
            results[name] = None
    
    return results, X_test, y_test

def analyze_results(results, X_test, y_test):
    """
    Детальный анализ результатов сравнения
    """
    print("\n" + "=" * 70)
    print("DETAILED RESULT ANALYSIS")
    print("=" * 70)
    
    analysis_data = []
    for name, result in results.items():
        if result is not None:
            analysis_data.append({
                'Model': name,
                'CV_Score': result['cv_mean'],
                'CV_Std': result['cv_std'],
                'Test_MSE': result['mse_test'],
                'Test_MAE': result['mae_test'],
                'Test_R2': result['r2_test']
            })
    
    df_results = pd.DataFrame(analysis_data)
    
    print("\n1. MODEL RANKING (CV score):")
    df_sorted_cv = df_results.sort_values('CV_Score', ascending=False)
    for i, (_, row) in enumerate(df_sorted_cv.iterrows(), 1):
        print(f"   {i:1d}. {row['Model']:25} : {row['CV_Score']:8.4f} ± {row['CV_Std']:.4f}")
    
    print("\n2. MODEL RANKING (test R²):")
    df_sorted_r2 = df_results.sort_values('Test_R2', ascending=False)
    for i, (_, row) in enumerate(df_sorted_r2.iterrows(), 1):
        print(f"   {i:1d}. {row['Model']:25} : R² = {row['Test_R2']:.3f}")
    
    print("\n3. STATISTICS:")
    best_cv_model = df_sorted_cv.iloc[0]
    best_r2_model = df_sorted_r2.iloc[0]
    
    print(f"   Best model by CV: {best_cv_model['Model']} ({best_cv_model['CV_Score']:.4f})")
    print(f"   Best model by test R²: {best_r2_model['Model']} ({best_r2_model['Test_R2']:.3f})")
    
    print("\n4. OVERFITTING ANALYSIS:")
    for name, result in results.items():
        if result is not None:
            cv_score = result['cv_mean']
            test_mse = -result['mse_test']
            overfitting_gap = cv_score - test_mse
            status = "⚠ ПЕРЕОБУЧЕНИЕ" if overfitting_gap > abs(cv_score) * 0.3 else "✓ НОРМА"
            print(f"   {name:25} : Gap = {overfitting_gap:7.4f} ({status})")
    
    return df_results

def plot_comparison(results, df_results):
    """
    Визуализация результатов сравнения
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Model performance comparison', fontsize=16, fontweight='bold')
    
    # 1. CV Scores
    models = df_results['Model']
    cv_scores = df_results['CV_Score']
    cv_stds = df_results['CV_Std']
    
    axes[0, 0].barh(models, cv_scores, xerr=cv_stds, alpha=0.7, color='skyblue')
    axes[0, 0].set_title('Cross-validation (neg MSE)')
    axes[0, 0].set_xlabel('Neg Mean Squared Error')
    axes[0, 0].grid(axis='x', alpha=0.3)
    
    # 2. R² Scores
    r2_scores = df_results['Test_R2']
    colors = ['green' if x > 0 else 'red' for x in r2_scores]
    axes[0, 1].barh(models, r2_scores, color=colors, alpha=0.7)
    axes[0, 1].set_title('R² on test set')
    axes[0, 1].set_xlabel('R² Score')
    axes[0, 1].axvline(x=0, color='black', linestyle='--', alpha=0.5)
    axes[0, 1].grid(axis='x', alpha=0.3)
    
    # 3. Test MSE
    test_mse = df_results['Test_MSE']
    axes[1, 0].barh(models, test_mse, alpha=0.7, color='lightcoral')
    axes[1, 0].set_title('MSE on test set')
    axes[1, 0].set_xlabel('Mean Squared Error')
    axes[1, 0].grid(axis='x', alpha=0.3)
    
    best_model_name = df_results.loc[df_results['Test_R2'].idxmax(), 'Model']
    best_result = results[best_model_name]
    
    axes[1, 1].scatter(best_result['y_test'], best_result['y_pred'], alpha=0.7, s=50)
    min_val = min(best_result['y_test'].min(), best_result['y_pred'].min())
    max_val = max(best_result['y_test'].max(), best_result['y_pred'].max())
    axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8)
    axes[1, 1].set_xlabel('True values')
    axes[1, 1].set_ylabel('Predicted values')
    axes[1, 1].set_title(f'Predictions vs true values\n({best_model_name})')
    axes[1, 1].grid(alpha=0.3)
    
    r2_best = best_result['r2_test']
    axes[1, 1].text(0.05, 0.95, f'R² = {r2_best:.3f}', 
                   transform=axes[1, 1].transAxes, fontsize=12,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    
    plt.tight_layout()
    plt.show()

def print_recommendations(df_results):
    """
    Вывод рекомендаций по выбору модели
    """
    print("\n" + "=" * 70)
    print("RECOMMENDATIONS")
    print("=" * 70)
    
    best_cv = df_results.loc[df_results['CV_Score'].idxmax()]
    best_r2 = df_results.loc[df_results['Test_R2'].idxmax()]
    best_stable = df_results.loc[df_results['CV_Std'].idxmin()]
    
    print("🎯 TOP MODELS:")
    print(f"   • By cross-validation: {best_cv['Model']} (score: {best_cv['CV_Score']:.4f})")
    print(f"   • By test R²: {best_r2['Model']} (R²: {best_r2['Test_R2']:.3f})")
    print(f"   • Most stable: {best_stable['Model']} (std: {best_stable['CV_Std']:.4f})")
    
    print("\n💡 RECOMMENDATIONS:")
    if best_r2['Test_R2'] < 0.3:
        print("   • Low model quality. Consider:")
        print("     - Collect more data")
        print("     - Engineer features")
        print("     - Try other algorithms")
    elif best_r2['Test_R2'] < 0.6:
        print("   • Moderate quality; useful for trend analysis.")
    else:
        print("   • Good quality; suitable for predictions.")
    
    print(f"\n📊 ОБЩАЯ ОЦЕНКА:")
    avg_r2 = df_results['Test_R2'].mean()
    if avg_r2 < 0:
        print("   • Quality below random guessing")
    elif avg_r2 < 0.3:
        print("   • Weak quality")
    elif avg_r2 < 0.6:
        print("   • Moderate quality") 
    elif avg_r2 < 0.8:
        print("   • Good quality")
    else:
        print("   • Excellent quality!")

print("RUNNING FULL MODEL BENCHMARK...")
results, X_test, y_test = compare_models_for_small_data(X[features], y)
df_results = analyze_results(results, X_test, y_test)
plot_comparison(results, df_results)
print_recommendations(df_results)

print("\n" + "=" * 70)
print("FULL RESULTS (TABLE)")
print("=" * 70)
print(df_results.round(4))

In [ ]:
def plot_extra_trees_predictions(results, model_name="Extra Trees"):
    """
    Рисует комплексный график распределения y_pred и y_test для указанной модели
    """
    if model_name not in results:
        print(f"Модель {model_name} не найдена в результатах")
        return
    
    result = results[model_name]
    y_test = result['y_test']
    y_pred = result['y_pred']
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'Prediction diagnostics: {model_name}', fontsize=16, fontweight='bold')
    
    axes[0, 0].scatter(y_test, y_pred, alpha=0.6, s=50, color='blue', edgecolors='black', linewidth=0.5)
    
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Ideal line')
    
    axes[0, 0].set_xlabel('True values (y_test)')
    axes[0, 0].set_ylabel('Predicted values (y_pred)')
    axes[0, 0].set_title('Predictions vs true values')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    r2 = result['r2_test']
    mse = result['mse_test']
    mae = result['mae_test']
    
    textstr = f'R² = {r2:.3f}\nMSE = {mse:.2f}\nMAE = {mae:.2f}'
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
    axes[0, 0].text(0.05, 0.95, textstr, transform=axes[0, 0].transAxes, fontsize=12,
                   verticalalignment='top', bbox=props)
    
    residuals = y_test - y_pred
    axes[0, 1].hist(residuals, bins=15, alpha=0.7, color='red', edgecolor='black')
    axes[0, 1].axvline(x=0, color='black', linestyle='--', linewidth=2)
    axes[0, 1].set_xlabel('Errors (y_test - y_pred)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Prediction error distribution')
    axes[0, 1].grid(True, alpha=0.3)
    
    mean_error = np.mean(residuals)
    std_error = np.std(residuals)
    textstr_err = f'Mean: {mean_error:.2f}\nStd: {std_error:.2f}'
    axes[0, 1].text(0.05, 0.95, textstr_err, transform=axes[0, 1].transAxes, fontsize=12,
                   verticalalignment='top', bbox=props)
    
    axes[1, 0].hist(y_test, bins=15, alpha=0.5, color='blue', label='y_test', edgecolor='black')
    axes[1, 0].hist(y_pred, bins=15, alpha=0.5, color='red', label='y_pred', edgecolor='black')
    axes[1, 0].set_xlabel('Values')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Distribution comparison: y_test vs y_pred')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    boxplot_data = [y_test, y_pred]
    axes[1, 1].boxplot(boxplot_data, labels=['y_test', 'y_pred'])
    axes[1, 1].set_ylabel('Values')
    axes[1, 1].set_title('Box plot: y_test vs y_pred')
    axes[1, 1].grid(True, alpha=0.3)
    
    q25_test, q75_test = np.percentile(y_test, [25, 75])
    q25_pred, q75_pred = np.percentile(y_pred, [25, 75])
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 ДЕТАЛЬНАЯ СТАТИСТИКА ДЛЯ {model_name}:")
    print("=" * 50)
    print(f"Качество модели:")
    print(f"  R² (test): {r2:.4f}")
    print(f"  MSE (test): {mse:.4f}")
    print(f"  MAE (test): {mae:.4f}")
    
    print(f"\nСтатистика y_test:")
    print(f"  Mean: {y_test.mean():.4f}")
    print(f"  Стандартное отклонение: {y_test.std():.4f}")
    print(f"  Минимум: {y_test.min():.4f}")
    print(f"  Максимум: {y_test.max():.4f}")
    
    print(f"\nСтатистика y_pred:")
    print(f"  Mean: {y_pred.mean():.4f}")
    print(f"  Стандартное отклонение: {y_pred.std():.4f}")
    print(f"  Минимум: {y_pred.min():.4f}")
    print(f"  Максимум: {y_pred.max():.4f}")
    
    print(f"\nСтатистика ошибок:")
    print(f"  Средняя ошибка: {mean_error:.4f}")
    print(f"  Std of errors: {std_error:.4f}")
    print(f"  Median error: {np.median(residuals):.4f}")
    
    return fig

In [ ]:
print("Plotting Extra Trees diagnostics...")
plot_extra_trees_predictions(results, "Extra Trees")

In [ ]:
print("Plotting Random Forest diagnostics...")
plot_extra_trees_predictions(results, "Random Forest ")

In [ ]:
def extra_trees_loo_analysis(X, y, test_size=5, random_state=42):
    """
    Полный анализ Extra Trees с Leave-One-Out и предсказаниями на новых данных
    """
    print("=" * 80)
    print("FULL EXTRA TREES ANALYSIS WITH LEAVE-ONE-OUT")
    print("=" * 80)
    
    X_train, X_new, y_train, y_new = train_test_split(
        X, y, test_size=test_size, random_state=random_state, shuffle=True
    )
    
    print(f"Размер тренировочных данных: {X_train.shape}")
    print(f"Размер новых данных (модель никогда не видела): {X_new.shape}")
    print(f"New образцы для предсказания: {list(X_new.index)}")
    
    et_model = ExtraTreesRegressor(
        n_estimators=30,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.3,
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )
    
    print("\n" + "=" * 50)
    print("LEAVE-ONE-OUT CROSS-VALIDATION")
    print("=" * 50)
    
    loo = LeaveOneOut()
    loo_scores = []
    loo_predictions = []
    loo_true_values = []
    
    for train_idx, test_idx in loo.split(X_train):
        X_train_fold, X_test_fold = X_train.iloc[train_idx], X_train.iloc[test_idx]
        y_train_fold, y_test_fold = y_train.iloc[train_idx], y_train.iloc[test_idx]
        
        et_model.fit(X_train_fold, y_train_fold)
        y_pred_fold = et_model.predict(X_test_fold)
        
        mse_fold = mean_squared_error(y_test_fold, y_pred_fold)
        loo_scores.append(-mse_fold)
        loo_predictions.extend(y_pred_fold)
        loo_true_values.extend(y_test_fold)
    
    print("\n" + "=" * 50)
    print("FINAL MODEL FIT")
    print("=" * 50)
    
    et_model.fit(X_train, y_train)
    
    y_train_pred = et_model.predict(X_train)
    
    y_new_pred = et_model.predict(X_new)
    
    train_r2 = r2_score(y_train, y_train_pred)
    train_mse = mean_squared_error(y_train, y_train_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    
    loo_r2 = r2_score(loo_true_values, loo_predictions)
    loo_mse = mean_squared_error(loo_true_values, loo_predictions)
    loo_mae = mean_absolute_error(loo_true_values, loo_predictions)
    
    new_r2 = r2_score(y_new, y_new_pred)
    new_mse = mean_squared_error(y_new, y_new_pred)
    new_mae = mean_absolute_error(y_new, y_new_pred)
    
    print("\n📊 RESULTS:")
    print("Metrics on training data:")
    print(f"  R²: {train_r2:.4f}, MSE: {train_mse:.4f}, MAE: {train_mae:.4f}")
    
    print("\nLeave-one-out metrics:")
    print(f"  R²: {loo_r2:.4f}, MSE: {loo_mse:.4f}, MAE: {loo_mae:.4f}")
    print(f"  LOO Score (mean neg_MSE): {np.mean(loo_scores):.4f} ± {np.std(loo_scores):.4f}")
    
    print("\nMetrics on NEW data (held out from training):")
    print(f"  R²: {new_r2:.4f}, MSE: {new_mse:.4f}, MAE: {new_mae:.4f}")
    
    plot_comprehensive_analysis(
        y_train, y_train_pred, 
        loo_true_values, loo_predictions,
        y_new, y_new_pred,
        X_new.index
    )
    
    print_new_predictions_details(X_new, y_new, y_new_pred, X_new.index)

    return {
        'model': et_model,
        'X_train': X_train,
        'y_train': y_train,
        'X_new': X_new,
        'y_new': y_new,
        'y_new_pred': y_new_pred,
        'loo_scores': loo_scores,
        'loo_predictions': loo_predictions,
        'loo_true_values': loo_true_values,
        'metrics': {
            'train': {'r2': train_r2, 'mse': train_mse, 'mae': train_mae},
            'loo': {'r2': loo_r2, 'mse': loo_mse, 'mae': loo_mae},
            'new': {'r2': new_r2, 'mse': new_mse, 'mae': new_mae}
        }
    }
    

def plot_comprehensive_analysis(y_train, y_train_pred, loo_true, loo_pred, y_new, y_new_pred, new_indices):
    """
    Комплексная визуализация результатов
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Extra Trees regressor — overview', fontsize=16, fontweight='bold')
    
    axes[0, 0].scatter(y_train, y_train_pred, alpha=0.6, color='blue', s=50)
    min_val = min(y_train.min(), y_train_pred.min())
    max_val = max(y_train.max(), y_train_pred.max())
    axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    axes[0, 0].set_xlabel('True values (train)')
    axes[0, 0].set_ylabel('Predicted values (train)')
    axes[0, 0].set_title('Parity plot: training data')
    axes[0, 0].grid(True, alpha=0.3)
    r2_train = r2_score(y_train, y_train_pred)
    axes[0, 0].text(0.05, 0.95, f'R² = {r2_train:.3f}', transform=axes[0, 0].transAxes,
                   bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    # 2. Parity Plot - LOO
    axes[0, 1].scatter(loo_true, loo_pred, alpha=0.6, color='green', s=50)
    min_val_loo = min(min(loo_true), min(loo_pred))
    max_val_loo = max(max(loo_true), max(loo_pred))
    axes[0, 1].plot([min_val_loo, max_val_loo], [min_val_loo, max_val_loo], 'r--', linewidth=2)
    axes[0, 1].set_xlabel('True values (LOO)')
    axes[0, 1].set_ylabel('Predicted values (LOO)')
    axes[0, 1].set_title('Parity Plot: Leave-One-Out')
    axes[0, 1].grid(True, alpha=0.3)
    r2_loo = r2_score(loo_true, loo_pred)
    axes[0, 1].text(0.05, 0.95, f'R² = {r2_loo:.3f}', transform=axes[0, 1].transAxes,
                   bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    axes[0, 2].scatter(y_new, y_new_pred, alpha=0.6, color='red', s=80, edgecolors='black')
    for i, idx in enumerate(new_indices):
        axes[0, 2].annotate(f'{idx}', (y_new.iloc[i], y_new_pred[i]), 
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
    min_val_new = min(min(y_new), min(y_new_pred))
    max_val_new = max(max(y_new), max(y_new_pred))
    axes[0, 2].plot([min_val_new, max_val_new], [min_val_new, max_val_new], 'r--', linewidth=2)
    axes[0, 2].set_xlabel('True values (new)')
    axes[0, 2].set_ylabel('Predicted values (new)')
    axes[0, 2].set_title('Parity plot: new data')
    axes[0, 2].grid(True, alpha=0.3)
    r2_new = r2_score(y_new, y_new_pred)
    axes[0, 2].text(0.05, 0.95, f'R² = {r2_new:.3f}', transform=axes[0, 2].transAxes,
                   bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    residuals_train = y_train - y_train_pred
    axes[1, 0].hist(residuals_train, bins=15, alpha=0.7, color='blue', edgecolor='black')
    axes[1, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[1, 0].set_xlabel('Residuals (train)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Residual distribution: training')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].text(0.05, 0.95, f'Mean: {residuals_train.mean():.3f}\nStd: {residuals_train.std():.3f}', 
                   transform=axes[1, 0].transAxes, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    residuals_loo = np.array(loo_true) - np.array(loo_pred)
    axes[1, 1].hist(residuals_loo, bins=15, alpha=0.7, color='green', edgecolor='black')
    axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[1, 1].set_xlabel('Residuals (LOO)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Residual distribution: LOO')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].text(0.05, 0.95, f'Mean: {residuals_loo.mean():.3f}\nStd: {residuals_loo.std():.3f}', 
                   transform=axes[1, 1].transAxes, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    residuals_new = y_new - y_new_pred
    axes[1, 2].hist(residuals_new, bins=min(10, len(y_new)), alpha=0.7, color='red', edgecolor='black')
    axes[1, 2].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[1, 2].set_xlabel('Residuals (new)')
    axes[1, 2].set_ylabel('Frequency')
    axes[1, 2].set_title('Residual distribution: new data')
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].text(0.05, 0.95, f'Mean: {residuals_new.mean():.3f}\nStd: {residuals_new.std():.3f}', 
                   transform=axes[1, 2].transAxes, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    plt.tight_layout()
    plt.show()

def print_new_predictions_details(X_new, y_new, y_new_pred, indices):
    """
    Детальная информация о предсказаниях на новых данных
    """
    print("\n" + "=" * 80)
    print("DETAILED PREDICTION REPORT (NEW DATA)")
    print("=" * 80)
    
    predictions_df = pd.DataFrame({
        'Sample index': indices,
        'True value': y_new.values,
        'Predicted value': y_new_pred,
        'Error': y_new.values - y_new_pred,
        'Absolute error': np.abs(y_new.values - y_new_pred),
        'Relative error (%)': np.abs((y_new.values - y_new_pred) / y_new.values) * 100
    })
    
    print(predictions_df.round(4))
    
    print(f"\n📈 СТАТИСТИКА ОШИБОК НА НОВЫХ ДАННЫХ:")
    print(f"  Mean absolute error: {predictions_df['Absolute error'].mean():.4f}")
    print(f"  Max absolute error: {predictions_df['Absolute error'].max():.4f}")
    print(f"  Mean relative error: {predictions_df['Relative error (%)'].mean():.2f}%")
    print(f"  Std of errors: {predictions_df['Error'].std():.4f}")

In [ ]:
print("Running Extra Trees diagnostics...")
results = extra_trees_loo_analysis(X[features], y, test_size=10, random_state=42)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)

final_model = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n=== PREDICTION RESULTS ===")
print(f"MSE на тесте: {mse:.4f}")
print(f"R² на тесте: {r2:.4f}")
print(f"Количество features: {len(features)}")

## Post-LOO diagnostics
Feature-importance extraction after the genetic / LOO benchmarking cells.


In [ ]:
def get_feature_importance_simple(model, feature_names):
    """
    Быстрое получение важности features
    """
    importance = model.feature_importances_
    
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(10, 8))
    top_20 = importance_df.head(20)
    
    plt.barh(top_20['feature'], top_20['importance'], color='lightblue')
    plt.xlabel('Feature importance')
    plt.title('Top 20 feature importances')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("Top-10 most important features:")
    print(top_20.head(10).to_string(index=False))
    
    return importance_df

In [ ]:
X[features]

In [ ]:
importance_df = get_feature_importance_simple(results["model"], X[features].columns)